# ManoVarta Aya DAIC Continue

This notebook mounts Drive, syncs the latest `main`, downloads the saved Aya adapter bundle, and continues extractor fine-tuning with `DAIC-WOZ` auxiliary supervision.

Checkpoints and reports are written directly into `MyDrive/ManoVartaOutputs` so the run can resume safely after interruptions.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.error
import urllib.request
import zipfile

drive.mount('/content/drive', force_remount=True)

REPO_DIR = Path('/content/ManoVarta')
REPO_URL = 'https://github.com/RitwijParmar/ManoVarta.git'
DRIVE_ROOT = Path('/content/drive/MyDrive')
OUT_ROOT = DRIVE_ROOT / 'ManoVartaOutputs'
ART_ROOT = Path('/content/aya_artifacts')
OUTER_ROOT = Path('/content/aya_outer')

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)

def find_daic_root(root: Path) -> Path:
    direct = root / 'DAIC-WOZ'
    if direct.exists():
        return direct
    for name in [
        'DAIC-WOZ',
        'DAIC_WOZ',
        'DAIC',
        'daic',
        'DAIC-WOZ_Depression',
        'DAICWOZ',
        'DAIC-WOZ_Depression_Dataset',
    ]:
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    csvs = sorted(root.rglob('train_split_Depression_AVEC2017.csv'))
    if csvs:
        return csvs[0].parent
    raise FileNotFoundError('Could not find DAIC-WOZ in MyDrive')

def ensure_daic_root(drive_root: Path, repo_dir: Path) -> Path:
    try:
        daic_root = find_daic_root(drive_root)
        print(f'Using DAIC root from Drive: {daic_root}')
        return daic_root
    except FileNotFoundError:
        runtime_root = Path('/content/DAIC-WOZ-transcripts')
        if (runtime_root / 'train_split_Depression_AVEC2017.csv').exists():
            print(f'Using existing runtime DAIC transcripts: {runtime_root}')
            return runtime_root
        print('DAIC-WOZ not found in Drive; downloading transcript-only fallback into runtime...')
        run(
            [
                sys.executable,
                'tools/fetch_daic_woz_transcripts.py',
                '--output-dir',
                str(runtime_root),
            ],
            cwd=repo_dir,
        )
        return runtime_root

def run_capture(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, text=True, capture_output=True)

def fallback_zip(repo_dir: Path, repo_url: str, branch: str = 'main'):
    repo_path = repo_url.removeprefix('https://github.com/').removesuffix('.git')
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        zip_url = f'https://api.github.com/repos/{repo_path}/zipball/{branch}'
    else:
        zip_url = f'https://github.com/{repo_path}/archive/refs/heads/{branch}.zip'
    request = urllib.request.Request(zip_url)
    if token:
        request.add_header('Authorization', f'Bearer {token}')
        request.add_header('Accept', 'application/vnd.github+json')
    with tempfile.TemporaryDirectory(prefix='manovarta-colab-') as tmp_dir_str:
        tmp_dir = Path(tmp_dir_str)
        archive_path = tmp_dir / 'repo.zip'
        with urllib.request.urlopen(request) as response, archive_path.open('wb') as handle:
            shutil.copyfileobj(response, handle)
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(tmp_dir)
        extracted_root = next(path for path in tmp_dir.iterdir() if path.is_dir())
        shutil.move(str(extracted_root), str(repo_dir))

def sync_repo(repo_dir: Path, repo_url: str, branch: str = 'main'):
    if (repo_dir / '.git').exists():
        for cmd in [
            ['git', '-C', str(repo_dir), 'fetch', 'origin'],
            ['git', '-C', str(repo_dir), 'checkout', branch],
            ['git', '-C', str(repo_dir), 'reset', '--hard', f'origin/{branch}'],
        ]:
            result = run_capture(cmd)
            if result.returncode != 0:
                print(result.stdout)
                print(result.stderr)
                raise subprocess.CalledProcessError(result.returncode, cmd)
        return
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    token = os.environ.get('GITHUB_TOKEN')
    clone_urls = []
    if token and repo_url.startswith('https://github.com/'):
        clone_urls.append(repo_url.replace('https://', f'https://x-access-token:{token}@'))
    clone_urls.append(repo_url)
    last_error = None
    for clone_url in clone_urls:
        result = run_capture(['git', 'clone', '--branch', branch, clone_url, str(repo_dir)])
        if result.returncode == 0:
            return
        last_error = result
        print(result.stdout)
        print(result.stderr)
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
    try:
        fallback_zip(repo_dir, repo_url, branch)
        return
    except Exception as exc:
        print(f'zip fallback failed: {exc}')
    if last_error is not None:
        raise subprocess.CalledProcessError(last_error.returncode, last_error.args)
    raise RuntimeError('Repo sync failed without a captured clone error')

sync_repo(REPO_DIR, REPO_URL)

run([sys.executable, 'tools/colab_bootstrap.py'], cwd=REPO_DIR)
run(['nvidia-smi'])

from huggingface_hub import hf_hub_download

ART_ROOT.mkdir(parents=True, exist_ok=True)
if OUTER_ROOT.exists():
    shutil.rmtree(OUTER_ROOT)
OUTER_ROOT.mkdir(parents=True, exist_ok=True)

bundle_path = ART_ROOT / 'aya_eval_upload.tar.gz'
bundle_url = os.environ.get('AYA_BUNDLE_URL', 'https://litter.catbox.moe/d6ggri.gz')
if bundle_path.exists():
    print(f'Using existing Aya bundle: {bundle_path}')
elif bundle_url:
    print(f'Downloading Aya bundle from fallback URL: {bundle_url}')
    urllib.request.urlretrieve(bundle_url, bundle_path)
else:
    try:
        bundle_path = Path(
            hf_hub_download(
                repo_id='ritwijar/manovarta-aya-eval-artifacts',
                filename='aya_eval_upload.tar.gz',
                repo_type='model',
                local_dir=str(ART_ROOT),
            )
        )
    except Exception as exc:
        raise RuntimeError(
            'Aya adapter download failed. Set AYA_BUNDLE_URL or log into Hugging Face and rerun this cell.'
        ) from exc

with tarfile.open(bundle_path, 'r:gz') as tf:
    tf.extractall(OUTER_ROOT)

adapter_dir = OUTER_ROOT / 'aya_bundle'
assert (adapter_dir / 'adapter_config.json').exists(), f'Missing Aya adapter under {adapter_dir}'

daic_root = ensure_daic_root(DRIVE_ROOT, REPO_DIR)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    'tools/run_colab_daic_continue.py',
    '--device', 'cuda',
    '--daic-root', str(daic_root),
    '--init-adapter', str(adapter_dir),
    '--drive-dir', str(OUT_ROOT),
    '--extractor-model', 'CohereLabs/aya-expanse-8b',
]

print('DAIC root:', daic_root)
print('Aya adapter:', adapter_dir)
print('Running:', ' '.join(map(str, cmd)))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
